# Análise Experimental — Extração Estruturada a partir de Comandos de Voz

Este notebook documenta e analisa o pipeline experimental de extração estruturada a partir de comandos de voz em português brasileiro para equipamentos médicos.

**Pipeline:**
```
Áudio → Whisper (STT) → LLM (Groq/LLaMA) → Pydantic → JSON validado
```

**Hipótese experimental:**  
Uma abordagem híbrida (LLM + regras de domínio médico) produz mais JSONs aderentes ao schema e menos erros críticos do que LLM puro.

**Variantes comparadas:**
- **Variante A:** LLM puro — Whisper + prompt + Pydantic, sem regras de domínio
- **Variante B:** Híbrida — mesma base + normalização de unidade + validação de faixa clínica + detecção de inferência

## 0. Setup e imports

In [1]:
import json
import sys
import os
import pandas as pd

sys.path.insert(0, os.path.join('..', 'src'))
sys.path.insert(0, '..')

# Carrega os resultados gerados pelos pipelines
with open('../data/transcripts/stt_evaluation.json', encoding='utf-8') as f:
    stt_data = json.load(f)

with open('../data/results/variante_a.json', encoding='utf-8') as f:
    var_a = json.load(f)

with open('../data/results/variante_b.json', encoding='utf-8') as f:
    var_b = json.load(f)

with open('../data/results/evaluation.json', encoding='utf-8') as f:
    evaluation = json.load(f)

print('Dados carregados com sucesso.')
print(f"STT: {stt_data['total_casos']} casos")
print(f"Variante A: {var_a['total_casos']} casos")
print(f"Variante B: {var_b['total_casos']} casos")

Dados carregados com sucesso.
STT: 25 casos
Variante A: 25 casos
Variante B: 25 casos


---
## 1. Análise de qualidade do STT

O Whisper (`base`, `language="pt"`) foi usado para transcrever os 25 áudios. Os áudios são de dois tipos:
- **Voz humana real** (10 casos): gravados pela candidata em `.m4a`
- **TTS sintético** (15 casos): gerados com `gTTS` em `.mp3`

As métricas são calculadas em duas versões:
- **Raw:** comparação direta entre referência e transcrição
- **Normalizado:** após normalização lexical (números por extenso → algarismos, unidades → forma canônica)

In [2]:
# Métricas globais de STT
metricas = stt_data['metricas']

df_metricas = pd.DataFrame([
    {'Métrica': 'WER', 'Raw': metricas['wer_raw_medio'], 'Normalizado': metricas['wer_normalizado_medio'],
     'Redução (%)': round((1 - metricas['wer_normalizado_medio'] / metricas['wer_raw_medio']) * 100, 1)},
    {'Métrica': 'CER', 'Raw': metricas['cer_raw_medio'], 'Normalizado': metricas['cer_normalizado_medio'],
     'Redução (%)': round((1 - metricas['cer_normalizado_medio'] / metricas['cer_raw_medio']) * 100, 1)},
])

print('=== Métricas globais de STT ===')
print(df_metricas.to_string(index=False))
print()
print('Interpretação: a redução de ~45% no WER e ~61% no CER após normalização')
print('evidencia que grande parte do erro era artefato de representação')
print('(por extenso vs. algarismo), não de reconhecimento real.')

=== Métricas globais de STT ===
Métrica    Raw  Normalizado  Redução (%)
    WER 0.5967       0.3273         45.1
    CER 0.3411       0.1337         60.8

Interpretação: a redução de ~45% no WER e ~61% no CER após normalização
evidencia que grande parte do erro era artefato de representação
(por extenso vs. algarismo), não de reconhecimento real.


In [3]:
# WER por fonte (voz humana vs gTTS) e por categoria
resultados_stt = stt_data['resultados']

df_stt = pd.DataFrame(resultados_stt)

print('=== WER normalizado médio por fonte ===')
print(df_stt.groupby('fonte')[['wer_normalizado', 'cer_normalizado']].mean().round(4))
print()
print('=== WER normalizado médio por categoria ===')
print(df_stt.groupby('categoria')[['wer_normalizado', 'cer_normalizado']].mean().round(4))

=== WER normalizado médio por fonte ===
            wer_normalizado  cer_normalizado
fonte                                       
gtts                 0.1789           0.0542
voz_humana           0.5500           0.2530

=== WER normalizado médio por categoria ===
                 wer_normalizado  cer_normalizado
categoria                                        
ambiguo                   0.5000           0.3051
erro_stt                  0.4833           0.1582
fora_de_faixa             0.2500           0.0744
incompleto                0.0000           0.0000
unidade_omitida           0.1875           0.0509
valido_simples            0.4167           0.1648


In [4]:
# Tabela completa caso a caso
df_stt_display = df_stt[[
    'id', 'categoria', 'fonte',
    'referencia', 'transcricao_whisper',
    'wer_raw', 'wer_normalizado'
]].copy()

df_stt_display.columns = [
    'ID', 'Categoria', 'Fonte',
    'Referência', 'Transcrição Whisper',
    'WER Raw', 'WER Norm'
]

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 30)
df_stt_display

,ID,Categoria,Fonte,Referência,Transcrição Whisper,WER Raw,WER Norm
0,1,valido_simples,voz_humana,aumentar frequência respiratória para vinte rpm,Aumentar frequência respirantória para vinte e repem.,0.5000,0.5000
1,2,valido_simples,voz_humana,definir volume corrente em quinhentos mililitros,Definível um ecorrente em 500 ml.,0.8333,0.5000
2,3,valido_simples,voz_humana,reduzir pressão inspiratória para quinze centímetros de ...,Reduzir a impressão inspiradora para 15 cm de água.,0.7500,1.0000
3,4,valido_simples,voz_humana,aumentar fio dois para sessenta por cento,"Almentar o fio 2,60%.",1.0000,0.5000
4,5,valido_simples,gtts,diminuir frequência para doze rpm,diminuir frequência para 12 RPM.,0.4000,0.0000
5,6,valido_simples,gtts,ajustar volume corrente para quatrocentos e cinquenta mi...,ajustar volume corrente para 450 ml.,0.5000,0.0000
6,7,unidade_omitida,voz_humana,aumentar frequência para vinte,Aumentar a frequência para o vinte.,0.7500,0.5000
7,8,unidade_omitida,gtts,definir fio dois em oitenta,Defineir Fio 2 em 80.,0.6000,0.2500
8,9,unidade_omitida,gtts,volume corrente quinhentos,Volume corrente 500.,0.3333,0.0000
9,10,unidade_omitida,gtts,pressão trinta,pressão 30,0.5000,0.0000


### 1.1 Erros de STT identificados

A análise qualitativa identificou 5 padrões de erro:

| Tipo de erro | Exemplo | Impacto na extração |
|---|---|---|
| Inserção de fonema em termo técnico | "respiratória" → "respirantória" | Baixo — LLM resiliente |
| Transcrição fonética de sigla | "rpm" → "repem", "FiO₂" → "F2" | Médio — LLM recupera parcialmente |
| Colapso em fala rápida | "ajustar para vinte" → "Azueta paraə vwan Ontiniz" | Alto — irrecuperável |
| Normalização automática de número | "quinhentos" → "500" | Nenhum — é uma feature |
| Autocorreção ortográfica | "corente" → "corrente" | Nenhum — melhora a extração |

**Caso especial — erro irrecuperável (caso 19):**  
O áudio "fio dois cem por cento" foi transcrito como "Fio 200%". O Whisper interpretou "dois cem" como "duzentos" em vez de "dois" + "cem". O valor correto (100) foi perdido na transcrição — nenhuma regra de domínio consegue recuperar informação corrompida na fonte.

---
## 2. Extração estruturada — Variante A (LLM puro)

A Variante A usa o LLM (LLaMA 3.3 70B via Groq) com `temperature=0` para extrair os campos do schema diretamente da transcrição do Whisper, sem nenhuma regra de domínio adicional.

In [5]:
# Constrói DataFrame da Variante A
rows_a = []
for r in var_a['resultados']:
    saida = r.get('saida') or {}
    rows_a.append({
        'id': r['id'],
        'categoria': r['categoria'],
        'fonte': r['fonte'],
        'transcript': r['transcript_whisper'],
        'intent': saida.get('intent'),
        'parameter': saida.get('parameter'),
        'value': saida.get('value'),
        'unit': saida.get('unit'),
        'status': saida.get('status'),
        'confidence': saida.get('confidence'),
        'requires_confirmation': saida.get('requires_confirmation'),
        'n_validation_errors': len(saida.get('validation_errors', [])),
        'erro_pipeline': r.get('erro'),
    })

df_a = pd.DataFrame(rows_a)

print('=== Variante A — distribuição de status ===')
print(df_a['status'].value_counts())
print()
print('=== Variante A — distribuição de confidence ===')
print(df_a['confidence'].value_counts())

=== Variante A — distribuição de status ===
status
ok               14
incompleto        7
fora_de_faixa     4
Name: count, dtype: int64

=== Variante A — distribuição de confidence ===
confidence
high    20
low      5
Name: count, dtype: int64


In [6]:
# Status por categoria
print('=== Variante A — status por categoria ===')
print(pd.crosstab(df_a['categoria'], df_a['status']))

=== Variante A — status por categoria ===
status           fora_de_faixa  incompleto  ok
categoria                                     
ambiguo                      0           4   0
erro_stt                     0           0   4
fora_de_faixa                4           0   0
incompleto                   0           3   0
unidade_omitida              0           0   4
valido_simples               0           0   6


---
## 3. Extração estruturada — Variante B (híbrida)

A Variante B aplica 4 regras determinísticas após a extração do LLM:
1. **Normalização de unidade** — mapeia sinônimos para a forma canônica
2. **Detecção de unidade inferida** — rebaixa `confidence` para `low` se a unidade não foi dita explicitamente
3. **Validação de faixa clínica** — força `status: fora_de_faixa` se o valor está fora da faixa permitida
4. **Consistência de status** — promove `ok` para `ambiguo` se há `validation_errors`

In [7]:
# Constrói DataFrame da Variante B (saida_final)
rows_b = []
for r in var_b['resultados']:
    saida = r.get('saida_final') or {}
    saida_llm = r.get('saida_llm') or {}
    rows_b.append({
        'id': r['id'],
        'categoria': r['categoria'],
        'fonte': r['fonte'],
        'transcript': r['transcript_whisper'],
        'intent': saida.get('intent'),
        'parameter': saida.get('parameter'),
        'value': saida.get('value'),
        'unit': saida.get('unit'),
        'status': saida.get('status'),
        'confidence': saida.get('confidence'),
        'requires_confirmation': saida.get('requires_confirmation'),
        'n_validation_errors': len(saida.get('validation_errors', [])),
        # campos para comparar LLM vs final
        'confidence_llm': saida_llm.get('confidence'),
        'req_conf_llm': saida_llm.get('requires_confirmation'),
    })

df_b = pd.DataFrame(rows_b)

print('=== Variante B — distribuição de status ===')
print(df_b['status'].value_counts())
print()
print('=== Variante B — distribuição de confidence ===')
print(df_b['confidence'].value_counts())

=== Variante B — distribuição de status ===
status
ok               14
incompleto        7
fora_de_faixa     4
Name: count, dtype: int64

=== Variante B — distribuição de confidence ===
confidence
low     14
high    11
Name: count, dtype: int64


In [8]:
# Casos onde a camada de regras mudou algo em relação ao LLM
alterados = df_b[
    (df_b['confidence'] != df_b['confidence_llm']) |
    (df_b['requires_confirmation'] != df_b['req_conf_llm'])
][['id', 'categoria', 'confidence_llm', 'confidence', 'req_conf_llm', 'requires_confirmation']]

alterados.columns = [
    'ID', 'Categoria',
    'Confidence (LLM)', 'Confidence (Final)',
    'ReqConf (LLM)', 'ReqConf (Final)'
]

print(f'=== Casos alterados pelas regras de domínio: {len(alterados)} ===')
print(alterados.to_string(index=False))

=== Casos alterados pelas regras de domínio: 9 ===
 ID       Categoria Confidence (LLM) Confidence (Final)  ReqConf (LLM)  ReqConf (Final)
  1  valido_simples             high                low          False             True
  7 unidade_omitida             high                low          False             True
  8 unidade_omitida             high                low          False             True
  9 unidade_omitida             high                low          False             True
 10 unidade_omitida             high                low          False             True
 12         ambiguo             high                low           True             True
 17      incompleto             high                low           True             True
 22        erro_stt             high                low          False             True
 24        erro_stt             high                low          False             True


---
## 4. Comparação experimental: Variante A vs Variante B

In [9]:
ev_a = evaluation['variante_a']
ev_b = evaluation['variante_b']

# Tabela comparativa de métricas globais
metricas_globais = [
    ('Taxa schema válido',  ev_a['taxa_schema_valido'],  ev_b['taxa_schema_valido']),
    ('Taxa erro crítico',   ev_a['taxa_erro_critico'],   ev_b['taxa_erro_critico']),
    ('Acurácia média',      ev_a['acuracia_media'],      ev_b['acuracia_media']),
    ('F1 macro intent',     ev_a['f1_macro_intent'],     ev_b['f1_macro_intent']),
]

df_global = pd.DataFrame(metricas_globais, columns=['Métrica', 'Variante A', 'Variante B'])
df_global['Delta'] = (df_global['Variante B'] - df_global['Variante A']).round(4)
df_global['Delta (%)'] = (df_global['Delta'] * 100).round(2).astype(str) + '%'

print('=== Comparação global: Variante A vs Variante B ===')
print(df_global.to_string(index=False))

=== Comparação global: Variante A vs Variante B ===
           Métrica  Variante A  Variante B  Delta Delta (%)
Taxa schema válido      1.0000      1.0000 0.0000      0.0%
 Taxa erro crítico      0.0400      0.0400 0.0000      0.0%
    Acurácia média      0.8343      0.8800 0.0457     4.57%
   F1 macro intent      0.5307      0.5307 0.0000      0.0%


In [10]:
# Acurácia por campo
campos = list(ev_a['acuracia_por_campo'].keys())
rows_campo = []
for campo in campos:
    va = ev_a['acuracia_por_campo'][campo]
    vb = ev_b['acuracia_por_campo'][campo]
    delta = round(vb - va, 4)
    rows_campo.append({
        'Campo': campo,
        'Variante A': f'{va:.0%}',
        'Variante B': f'{vb:.0%}',
        'Delta': f'{delta:+.0%}',
    })

df_campos = pd.DataFrame(rows_campo)
print('=== Acurácia por campo ===')
print(df_campos.to_string(index=False))

=== Acurácia por campo ===
                Campo Variante A Variante B Delta
               intent        84%        84%   +0%
            parameter       100%       100%   +0%
                value        92%        92%   +0%
                 unit        84%        84%   +0%
               status        80%        80%   +0%
requires_confirmation        84%        88%   +4%
           confidence        60%        88%  +28%


In [11]:
# F1 por intent
intents = list(ev_a['f1_por_intent'].keys())
rows_f1 = []
for intent in intents:
    fa = ev_a['f1_por_intent'][intent]
    fb = ev_b['f1_por_intent'][intent]
    rows_f1.append({
        'Intent': intent,
        'Precision A': fa['precision'],
        'Recall A': fa['recall'],
        'F1 A': fa['f1'],
        'Precision B': fb['precision'],
        'Recall B': fb['recall'],
        'F1 B': fb['f1'],
    })

df_f1 = pd.DataFrame(rows_f1)
print('=== F1 por intent ===')
print(df_f1.to_string(index=False))
print()
print('Nota: consultar_parametro e desconhecido têm F1=0 porque aparecem')
print('raramente no dataset de 25 casos — o macro F1 é penalizado por classes raras.')

=== F1 por intent ===
             Intent  Precision A  Recall A   F1 A  Precision B  Recall B   F1 B
 aumentar_parametro         1.00    0.7143 0.8333         1.00    0.7143 0.8333
  reduzir_parametro         0.75    1.0000 0.8571         0.75    1.0000 0.8571
  definir_parametro         1.00    0.9286 0.9630         1.00    0.9286 0.9630
consultar_parametro         0.00    0.0000 0.0000         0.00    0.0000 0.0000
       desconhecido         0.00    0.0000 0.0000         0.00    0.0000 0.0000

Nota: consultar_parametro e desconhecido têm F1=0 porque aparecem
raramente no dataset de 25 casos — o macro F1 é penalizado por classes raras.


---
## 5. Análise de erros por categoria

In [12]:
from data.answer import GABARITO_POR_ID

# Constrói tabela de acertos/erros por campo e por categoria para Variante A
detalhes_a = ev_a['detalhes']
detalhes_b = ev_b['detalhes']

campos_avaliados = ['intent', 'parameter', 'value', 'unit', 'status',
                    'requires_confirmation', 'confidence']

rows_err = []
for det_a, det_b in zip(detalhes_a, detalhes_b):
    caso_id = det_a['id']
    cat = det_a['categoria']
    for campo in campos_avaliados:
        rows_err.append({
            'id': caso_id,
            'categoria': cat,
            'campo': campo,
            'correto_a': det_a['acertos'][campo],
            'correto_b': det_b['acertos'][campo],
        })

df_err = pd.DataFrame(rows_err)

# Acurácia por categoria e variante
acc_cat_a = df_err.groupby('categoria')['correto_a'].mean().round(3)
acc_cat_b = df_err.groupby('categoria')['correto_b'].mean().round(3)

df_cat = pd.DataFrame({'Variante A': acc_cat_a, 'Variante B': acc_cat_b})
df_cat['Delta'] = (df_cat['Variante B'] - df_cat['Variante A']).round(3)

print('=== Acurácia média por categoria ===')
print(df_cat.to_string())

=== Acurácia média por categoria ===
                 Variante A  Variante B  Delta
categoria                                     
ambiguo               0.607       0.643  0.036
erro_stt              0.857       0.857  0.000
fora_de_faixa         0.929       0.929  0.000
incompleto            0.810       0.857  0.047
unidade_omitida       0.714       1.000  0.286
valido_simples        1.000       0.952 -0.048


In [13]:
# Casos com erro crítico
erros_criticos = [
    d for d in detalhes_a if d['erro_critico']
]

print(f'=== Erros críticos (ambas as variantes): {len(erros_criticos)} ===')
for e in erros_criticos:
    caso_id = e['id']
    gab = GABARITO_POR_ID[caso_id]
    print(f"  Caso {caso_id} ({e['categoria']}): {gab['referencia']}")
    print(f"  Gabarito: value={gab['esperado']['value']}, status={gab['esperado']['status']}")
    
    # Busca transcrição e saída do pipeline A
    res_a = next(r for r in var_a['resultados'] if r['id'] == caso_id)
    print(f"  Whisper transcreveu: '{res_a['transcript_whisper']}'")
    print(f"  LLM extraiu: value={res_a['saida']['value']}, status={res_a['saida']['status']}")
    print(f"  Diagnóstico: erro irrecuperável na transcrição STT")

=== Erros críticos (ambas as variantes): 1 ===
  Caso 19 (fora_de_faixa): fio dois cem por cento
  Gabarito: value=100.0, status=ok
  Whisper transcreveu: 'Fio 200%'
  LLM extraiu: value=200.0, status=fora_de_faixa
  Diagnóstico: erro irrecuperável na transcrição STT


---
## 6. Exemplos de saída — casos selecionados

In [14]:
def mostrar_caso(caso_id, variante='b'):
    """Exibe a saída completa de um caso específico."""
    gab = GABARITO_POR_ID[caso_id]
    dados = var_b if variante == 'b' else var_a
    res = next(r for r in dados['resultados'] if r['id'] == caso_id)
    saida = res.get('saida_final') or res.get('saida')
    
    print(f"{'='*60}")
    print(f"Caso {caso_id} — {gab['categoria']}")
    print(f"Referência : {gab['referencia']}")
    print(f"Whisper    : {res['transcript_whisper']}")
    print(f"{'─'*60}")
    print(f"Saída (Variante {'B' if variante == 'b' else 'A'}):")
    for k, v in saida.items():
        if k != 'notes':
            print(f"  {k:<25}: {v}")
    print(f"{'─'*60}")
    print(f"Gabarito esperado:")
    for k, v in gab['esperado'].items():
        print(f"  {k:<25}: {v}")

# Caso válido simples
mostrar_caso(1)

Caso 1 — valido_simples
Referência : aumentar frequência respiratória para vinte rpm
Whisper    : Aumentar frequência respirantória para vinte e repem.
────────────────────────────────────────────────────────────
Saída (Variante B):
  intent                   : aumentar_parametro
  parameter                : frequencia_respiratoria
  value                    : 20.0
  unit                     : rpm
  status                   : ok
  confidence               : low
  requires_confirmation    : True
  validation_errors        : []
  normalized_transcript    : Aumentar frequência respiratória para 20 rpm
────────────────────────────────────────────────────────────
Gabarito esperado:
  intent                   : aumentar_parametro
  parameter                : frequencia_respiratoria
  value                    : 20.0
  unit                     : rpm
  status                   : ok
  confidence               : high
  requires_confirmation    : False
  validation_errors        : []


In [15]:
# Caso com unidade omitida — onde a Variante B corrige confidence
mostrar_caso(7)

Caso 7 — unidade_omitida
Referência : aumentar frequência para vinte
Whisper    : Aumentar a frequência para o vinte.
────────────────────────────────────────────────────────────
Saída (Variante B):
  intent                   : aumentar_parametro
  parameter                : frequencia_respiratoria
  value                    : 20.0
  unit                     : rpm
  status                   : ok
  confidence               : low
  requires_confirmation    : True
  validation_errors        : []
  normalized_transcript    : Aumentar a frequência para o vinte
────────────────────────────────────────────────────────────
Gabarito esperado:
  intent                   : aumentar_parametro
  parameter                : frequencia_respiratoria
  value                    : 20.0
  unit                     : rpm
  status                   : ok
  confidence               : low
  requires_confirmation    : True
  validation_errors        : []


In [16]:
# Caso ambíguo
mostrar_caso(13)

Caso 13 — ambiguo
Referência : fio dois tá baixo
Whisper    : Fio 2 tabaixo
────────────────────────────────────────────────────────────
Saída (Variante B):
  intent                   : reduzir_parametro
  parameter                : fio2
  value                    : None
  unit                     : %
  status                   : incompleto
  confidence               : low
  requires_confirmation    : True
  validation_errors        : ['valor não especificado']
  normalized_transcript    : Fio 2 abaixo
────────────────────────────────────────────────────────────
Gabarito esperado:
  intent                   : aumentar_parametro
  parameter                : fio2
  value                    : None
  unit                     : None
  status                   : ambiguo
  confidence               : low
  requires_confirmation    : True
  validation_errors        : ['valor não especificado']


In [17]:
# Caso fora de faixa
mostrar_caso(18)

Caso 18 — fora_de_faixa
Referência : definir frequência respiratória para trezentos rpm
Whisper    : Definir frequência respiratória para 300 RPM.
────────────────────────────────────────────────────────────
Saída (Variante B):
  intent                   : definir_parametro
  parameter                : frequencia_respiratoria
  value                    : 300.0
  unit                     : rpm
  status                   : fora_de_faixa
  confidence               : high
  requires_confirmation    : True
  validation_errors        : ['Valor fora da faixa normal (8-35)', 'valor 300.0 fora da faixa permitida (8–35 rpm)']
  normalized_transcript    : Definir frequência respiratória para 300 RPM
────────────────────────────────────────────────────────────
Gabarito esperado:
  intent                   : definir_parametro
  parameter                : frequencia_respiratoria
  value                    : 300.0
  unit                     : rpm
  status                   : fora_de_faixa
  confidenc

In [18]:
# Caso com erro de STT irrecuperável
mostrar_caso(19)

Caso 19 — fora_de_faixa
Referência : fio dois cem por cento
Whisper    : Fio 200%
────────────────────────────────────────────────────────────
Saída (Variante B):
  intent                   : definir_parametro
  parameter                : fio2
  value                    : 200.0
  unit                     : %
  status                   : fora_de_faixa
  confidence               : high
  requires_confirmation    : True
  validation_errors        : ['Valor fora da faixa normal (21-100)', 'valor 200.0 fora da faixa permitida (21–100 %)']
  normalized_transcript    : Fio 200%
────────────────────────────────────────────────────────────
Gabarito esperado:
  intent                   : definir_parametro
  parameter                : fio2
  value                    : 100.0
  unit                     : %
  status                   : ok
  confidence               : high
  requires_confirmation    : True
  validation_errors        : []


In [19]:
# Caso com colapso total de transcrição (voz humana, fala rápida)
mostrar_caso(14)

Caso 14 — ambiguo
Referência : ajustar para vinte
Whisper    : Azueta paraə vwan Ontiniz
────────────────────────────────────────────────────────────
Saída (Variante B):
  intent                   : desconhecido
  parameter                : None
  value                    : None
  unit                     : None
  status                   : incompleto
  confidence               : low
  requires_confirmation    : True
  validation_errors        : ['Transcrição não contém informações médicas reconhecíveis']
  normalized_transcript    : Azueta para vwan Ontiniz
────────────────────────────────────────────────────────────
Gabarito esperado:
  intent                   : definir_parametro
  parameter                : None
  value                    : 20.0
  unit                     : None
  status                   : ambiguo
  confidence               : low
  requires_confirmation    : True
  validation_errors        : ['parâmetro não identificado']


---
## 7. Síntese e conclusões

### 7.1 Qualidade do STT

O Whisper `base` com `language="pt"` apresentou WER normalizado médio de **0.3273** e CER normalizado de **0.1337**. A diferença entre métricas raw e normalizadas (−45% e −61% respectivamente) demonstra que o modelo está semanticamente correto em mais casos do que as métricas brutas indicam — a normalização lexical é etapa obrigatória antes de qualquer avaliação de qualidade de STT.

A voz humana apresentou WER significativamente maior que o TTS sintético, evidenciando que o modelo `base` sem fine-tuning é sensível a variações de prosódia, sotaque e velocidade de fala. Em ambiente hospitalar real, um modelo especializado ou fine-tuned seria necessário.

### 7.2 Extração estruturada

O LLM (LLaMA 3.3 70B) demonstrou robustez notável a transcrições ruins: em casos como "Definível um ecorrente em 500 ml" e "Almentar o fio 2,60%", extraiu os campos corretamente apesar da má qualidade do STT. A acurácia em `parameter` foi de 100% em ambas as variantes.

### 7.3 Comparação experimental

A hipótese foi parcialmente confirmada:

- **Confirmada:** a Variante B melhora significativamente `confidence` (+28pp) e `requires_confirmation` (+4pp)
- **Não confirmada:** a taxa de erro crítico permaneceu igual (0.04) — o único erro crítico é irrecuperável por qualquer camada de pós-processamento

O resultado central é que **regras determinísticas complementam o LLM exatamente onde ele é cego**: o modelo infere unidades corretamente mas não sinaliza que fez uma inferência. Em um sistema médico, essa distinção é clinicamente relevante — um parâmetro inferido deve ser confirmado pelo operador antes de ser aplicado.

### 7.4 Limitações

1. **Dataset pequeno (25 casos):** F1 de classes raras (`consultar_parametro`, `desconhecido`) é estatisticamente não confiável
2. **Whisper base:** vocabulário médico limitado; fine-tuning reduziria WER em termos técnicos
3. **Erro irrecuperável por STT:** corrupção de valor numérico na transcrição não é recuperável downstream
4. **Groq/LLaMA em vez de OpenAI:** troca motivada por ausência de créditos; resultados podem diferir em casos de borda

In [20]:
# Resumo final em tabela
resumo = {
    'WER médio (normalizado)': f"{stt_data['metricas']['wer_normalizado_medio']:.4f}",
    'CER médio (normalizado)': f"{stt_data['metricas']['cer_normalizado_medio']:.4f}",
    'Taxa schema válido (A)': f"{ev_a['taxa_schema_valido']:.0%}",
    'Taxa schema válido (B)': f"{ev_b['taxa_schema_valido']:.0%}",
    'Acurácia média (A)': f"{ev_a['acuracia_media']:.2%}",
    'Acurácia média (B)': f"{ev_b['acuracia_media']:.2%}",
    'Acurácia confidence (A)': f"{ev_a['acuracia_por_campo']['confidence']:.0%}",
    'Acurácia confidence (B)': f"{ev_b['acuracia_por_campo']['confidence']:.0%}",
    'Taxa erro crítico (A e B)': f"{ev_a['taxa_erro_critico']:.0%}",
}

print('=== RESUMO FINAL ===')
for k, v in resumo.items():
    print(f'  {k:<35}: {v}')

=== RESUMO FINAL ===
  WER médio (normalizado)            : 0.3273
  CER médio (normalizado)            : 0.1337
  Taxa schema válido (A)             : 100%
  Taxa schema válido (B)             : 100%
  Acurácia média (A)                 : 83.43%
  Acurácia média (B)                 : 88.00%
  Acurácia confidence (A)            : 60%
  Acurácia confidence (B)            : 88%
  Taxa erro crítico (A e B)          : 4%
